# 05 · Feature Engineering (Exploratory Only)

**IMPORTANT:** This notebook is **exploratory only**.
The official pipeline is managed by `dvc.yaml` and `dvc repro`.

---

**Input:** `data/processed/train_clean.csv` | `val_clean.csv` | `test_clean.csv` (DVC-tracked)

**Output:** `data/processed/train_feat.csv` | `val_feat.csv` | `test_feat.csv` (DVC-tracked)

**Config:** `configs/data_config.yaml` → `eda_derived.engineered_features`

---

### What this notebook does (Exploratory)

| Step | Action | Source / Note |
| :--- | :--- | :--- |
| 1 | Setup environment, imports, and `PROJECT_DIR` | `src.utils.paths` |
| 2 | Load cleaned splits from `data/processed/` | `DataLoader` + empty checks |
| 3 | Verify feature config exists in YAML | `data_config.yaml` → `eda_derived.engineered_features` |
| 4 | Run feature engineering (ratios + distances + drop) | `run_feature_engineering()` |
| 5 | Verify results (new features, nulls, inf, shapes) | Exploratory validation only |
| 6 | Feature statistics | Quick stats on new features |
| 7 | Schema consistency | All splits have same columns |

---

### FIT/TRANSFORM Rule

**No fitting/statistical learning happens here.**
Feature engineering applies deterministic transformations (ratios, distances) to all splits equally.
Therefore, this stage does **not** introduce train/validation/test leakage.

---

### Important Notes

- This notebook is **exploratory only** — DVC tracking is handled by the official pipeline (`dvc repro`).
- All transformations are **deterministic** — no train-only statistics are computed.
- Raw count columns are dropped after ratio creation to reduce multicollinearity.

---
## 0 - Setup & Load

In [50]:
# ===================================================================
# Section 0 · Setup
# ===================================================================
import os
import sys
from pathlib import Path

from src.utils.paths import PROJECT_DIR

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import importlib
importlib.invalidate_caches()

print(f"✅ Working dir : {os.getcwd()}")
print(f"✅ sys.path[0] : {sys.path[0]}")

✅ Working dir : /content/california_housing_full_project
✅ sys.path[0] : /content/california_housing_full_project


In [51]:
import logging
import pandas as pd
import numpy as np

from src.utils.logger import setup_logging, get_logger
from src.data.data_loader import DataLoader
from src.features.engineering import (
    run_feature_engineering,
    load_feature_config,
    EngineeringResult,
)

setup_logging(level=logging.INFO)
logger = get_logger("notebook.05_feature_engineering")

CONFIG_PATH = "configs/data_config.yaml"

print("Imports ready")

Imports ready


---
## 2 - Load Cleaned Splits

> We load all three splits because `run_feature_engineering()` applies the

> same pure-math transforms to each one independently.
> There is no fit step here - no leakage risk.

In [52]:
loader = DataLoader()

train = loader.load_processed("train_clean.csv")
val = loader.load_processed("val_clean.csv")
test = loader.load_processed("test_clean.csv")

if train is None or train.empty:
    raise ValueError("train_clean.csv is empty or None")
if val is None or val.empty:
    raise ValueError("val_clean.csv is empty or None")
if test is None or test.empty:
    raise ValueError("test_clean.csv is empty or None")

print(f"train : {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"val   : {val.shape[0]:,} rows x {val.shape[1]} cols")
print(f"test  : {test.shape[0]:,} rows x {test.shape[1]} cols")
print()
print("Train columns:")
print(train.columns.tolist())

2026-09-03 00:01:18 | INFO     | src.data.data_loader | DataLoader initialized | Drive mode: True
2026-09-03 00:01:18 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/processed/train_clean.csv
2026-09-03 00:01:18 | INFO     | src.data.data_loader | Loaded 'train_clean.csv' | shape=(14448, 11) | stage=processed
2026-09-03 00:01:18 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/processed/val_clean.csv
2026-09-03 00:01:18 | INFO     | src.data.data_loader | Loaded 'val_clean.csv' | shape=(3096, 11) | stage=processed
2026-09-03 00:01:18 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/processed/test_clean.csv
2026-09-03 00:01:18 | INFO     | src.data.data_loader | Loaded 'test_clean.csv' | shape=(3096, 11) | stage=processed
train : 14,448 rows x 11 cols
val   : 3,096 rows x 11 cols
test  : 3,096 rows x 11 cols

Train columns:
['longitude', 'latitude', 'housing_med

---
## 3  - Verify Feature Config

Confirm that `data_config.yaml` has the `eda_derived.engineered_features`
section. If `source = fallback`, the module will use hardcoded defaults
(which mirror the EDA results, but the YAML is the source of truth).

In [53]:
feat_cfg = load_feature_config(CONFIG_PATH)

print(f"Config source  : {feat_cfg.source}")
print()
print("Ratios to create:")
for name, formula in feat_cfg.ratios.items():
    print(f"  {name:<30} = {formula}")
print()
print("Distances to create:")
for name, hub in feat_cfg.distances.items():
    print(f"  {name:<15} -> lat={hub['lat']}, lon={hub['lon']}")
print()
print("Columns to drop after engineering:")
for col in feat_cfg.drop_cols:
    print(f"  {col}")
print()
if feat_cfg.source == "fallback":
    print("WARNING: eda_derived.engineered_features missing from data_config.yaml")
    print("Using hardcoded fallback defaults (values match last EDA run).")
else:
    print("Config loaded from data_config.yaml - ready to engineer")

2026-09-03 00:01:39 | INFO     | src.features.engineering | Feature config loaded: 3 ratios, 2 distances, 4 columns to drop.
Config source  : config

Ratios to create:
  rooms_per_household            = total_rooms / households
  bedrooms_per_room              = total_bedrooms / total_rooms
  population_per_household       = population / households

Distances to create:
  dist_SF         -> lat=37.77, lon=-122.42
  dist_LA         -> lat=34.05, lon=-118.24

Columns to drop after engineering:
  total_rooms
  total_bedrooms
  population
  households

Config loaded from data_config.yaml - ready to engineer



---
## 4 · Run Feature Engineering

`run_feature_engineering()` applies three steps in order:

1. **Ratio features** — divide count columns to reduce multicollinearity
   - `rooms_per_household` = `total_rooms / households`
   - `bedrooms_per_room` = `total_bedrooms / total_rooms`
   - `population_per_household` = `population / households`

2. **Distance features** — Euclidean distance to SF and LA price hubs
   - `dist_SF` → (37.77, -122.42)
   - `dist_LA` → (34.05, -118.24)

3. **Drop raw columns** — remove the raw size columns replaced by ratios
   - `total_rooms`, `total_bedrooms`, `population`, `households`

> ⚠️ **Zero denominators** produce `NaN` instead of `infinity` (safe division).

In [54]:
result = run_feature_engineering(
    train=train,
    val=val,
    test=test,
    config_path=CONFIG_PATH,
)

print(result.summary())

2026-09-03 00:02:11 | INFO     | src.features.engineering | ============================================================
2026-09-03 00:02:11 | INFO     | src.features.engineering | FEATURE ENGINEERING STARTED
2026-09-03 00:02:11 | INFO     | src.features.engineering | ============================================================
2026-09-03 00:02:11 | INFO     | src.features.engineering | Feature config loaded: 3 ratios, 2 distances, 4 columns to drop.
2026-09-03 00:02:11 | INFO     | src.features.engineering | Step 1/3 — Creating ratio features
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'rooms_per_household' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'bedrooms_per_room' | nulls=0


2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'population_per_household' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'rooms_per_household' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'bedrooms_per_room' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'population_per_household' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'rooms_per_household' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'bedrooms_per_room' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created ratio 'population_per_household' | nulls=0
2026-09-03 00:02:11 | INFO     | src.features.engineering | Step 2/3 — Creating distance features
2026-09-03 00:02:11 | INFO     | src.features.engineering | Created distance 'dist_SF' to (37.77, -122.42)
2026-09-03 00:02:11 | INFO     | src.featu

---
## 5 · Verify Results

**Five checks:**

1. **New features present** in all splits
2. **Raw columns dropped** from all splits
3. **Null check** — no NaN in new features (except from zero denominators)
4. **Infinity check** — no `inf` values in ratio features
5. **Shape consistency** — all splits have the same columns

In [55]:
print("-- New features check --")
new_cols = list(feat_cfg.ratios.keys()) + list(feat_cfg.distances.keys())

for col in new_cols:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        status = "OK" if col in df.columns else "MISSING"
        print(f"  {col:<30} in {name:<6} : {status}")

-- New features check --
  rooms_per_household            in train  : OK
  rooms_per_household            in val    : OK
  rooms_per_household            in test   : OK
  bedrooms_per_room              in train  : OK
  bedrooms_per_room              in val    : OK
  bedrooms_per_room              in test   : OK
  population_per_household       in train  : OK
  population_per_household       in val    : OK
  population_per_household       in test   : OK
  dist_SF                        in train  : OK
  dist_SF                        in val    : OK
  dist_SF                        in test   : OK
  dist_LA                        in train  : OK
  dist_LA                        in val    : OK
  dist_LA                        in test   : OK


In [56]:
print("-- Dropped columns check --")
for col in feat_cfg.drop_cols:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        still_present = col in df.columns
        status = "STILL PRESENT (unexpected)" if still_present else "OK (dropped)"
        print(f"  {col:<20} in {name:<6} : {status}")

-- Dropped columns check --
  total_rooms          in train  : OK (dropped)
  total_rooms          in val    : OK (dropped)
  total_rooms          in test   : OK (dropped)
  total_bedrooms       in train  : OK (dropped)
  total_bedrooms       in val    : OK (dropped)
  total_bedrooms       in test   : OK (dropped)
  population           in train  : OK (dropped)
  population           in val    : OK (dropped)
  population           in test   : OK (dropped)
  households           in train  : OK (dropped)
  households           in val    : OK (dropped)
  households           in test   : OK (dropped)


In [57]:
print("-- Null check in new features --")
for col in new_cols:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        if col not in df.columns:
            continue
        nulls = df[col].isnull().sum()
        status = "OK" if nulls == 0 else f"WARN ({nulls} nulls)"
        print(f"  {col:<30} {name:<6} : {status}")

-- Null check in new features --
  rooms_per_household            train  : OK
  rooms_per_household            val    : OK
  rooms_per_household            test   : OK
  bedrooms_per_room              train  : OK
  bedrooms_per_room              val    : OK
  bedrooms_per_room              test   : OK
  population_per_household       train  : OK
  population_per_household       val    : OK
  population_per_household       test   : OK
  dist_SF                        train  : OK
  dist_SF                        val    : OK
  dist_SF                        test   : OK
  dist_LA                        train  : OK
  dist_LA                        val    : OK
  dist_LA                        test   : OK


In [58]:
import numpy as np

print("-- Inf check in ratio features --")
for col in feat_cfg.ratios.keys():
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        if col not in df.columns:
            continue
        n_inf = np.isinf(df[col]).sum()
        status = "OK" if n_inf == 0 else f"WARN ({n_inf} inf values)"
        print(f"  {col:<30} {name:<6} : {status}")

-- Inf check in ratio features --
  rooms_per_household            train  : OK
  rooms_per_household            val    : OK
  rooms_per_household            test   : OK
  bedrooms_per_room              train  : OK
  bedrooms_per_room              val    : OK
  bedrooms_per_room              test   : OK
  population_per_household       train  : OK
  population_per_household       val    : OK
  population_per_household       test   : OK


---
## 6 - Feature Statistics

Quick stats on the new features - train only.

In [59]:
print("-- New feature statistics (train) --")
stats = result.train[new_cols].describe().T[
    ["mean", "std", "min", "max"]
].round(4)
print(stats.to_string())

-- New feature statistics (train) --
                            mean      std     min        max
rooms_per_household       5.4405   2.4715  0.8462   141.9091
bedrooms_per_room         0.2133   0.0650  0.0369     2.8052
population_per_household  3.0938  11.3626  0.6923  1243.3333
dist_SF                   3.8555   2.5041  0.0000     9.3079
dist_LA                   2.6726   2.4185  0.0000     9.8600


---
## 7 - Schema Consistency

In [60]:
print("-- Shape check --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    print(f"  {name:<6} : {df.shape[0]:,} rows x {df.shape[1]} cols")

print()
print("-- Column consistency --")
train_cols = list(result.train.columns)
for name, df in [("val", result.val), ("test", result.test)]:
    if list(df.columns) == train_cols:
        print(f"  {name} columns match train - OK")
    else:
        diff = set(train_cols) ^ set(df.columns)
        print(f"  {name} column mismatch: {diff}")

print()
print("-- Final column list --")
print(result.train.columns.tolist())

-- Shape check --
  train  : 14,448 rows x 12 cols
  val    : 3,096 rows x 12 cols
  test   : 3,096 rows x 12 cols

-- Column consistency --
  val columns match train - OK
  test columns match train - OK

-- Final column list --
['longitude', 'latitude', 'housing_median_age', 'median_income', 'median_house_value', 'ocean_proximity', 'lof_outlier', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'dist_SF', 'dist_LA']


## Summary & Next Steps

| Done | Details |
| :--- | :--- |
| Ratio features | `rooms_per_household`, `bedrooms_per_room`, `population_per_household` |
| Distance features | `dist_SF`, `dist_LA` |
| Raw cols dropped | `total_rooms`, `total_bedrooms`, `population`, `households` |
| Files saved | `data/processed/*_feat.csv` (saved but not DVC-tracked in this notebook) |
